# Survey Test — Foto-inspectie per taak

Haalt de taken op van de meest recente respondent en toont de bijbehorende foto's.
Gebruik dit om intersection-ID's te achterhalen die je wilt uitsluiten van de analyse.

In [ ]:
import sqlite3
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

# Paths relative to this notebook (which lives next to prepare_survey_images_new.py)
DB_PATH    = os.path.join('surveys', 'survey-intersection-safety', 'database.db')
ASSETS_DIR = os.path.join('surveys', 'survey-intersection-safety')

In [ ]:
def query(sql, params=()):
    """Run a read query and return results as a list of dicts."""
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        cur = conn.execute(sql, params)
        return [dict(r) for r in cur.fetchall()]


def url_to_path(url):
    """Convert /assets/images/xxx.jpeg to the local file path."""
    # url looks like /assets/images/182274007.jpeg
    return os.path.join(ASSETS_DIR, url.lstrip('/'))


def extract_id(url):
    """Extract the numeric intersection ID from an image URL."""
    match = re.search(r'(\d+)\.\w+$', url or '')
    return match.group(1) if match else 'onbekend'

In [ ]:
# Get the most recent respondent and their set_id
respondents = query(
    "SELECT respondent_id, set_id FROM Response ORDER BY respondent_id DESC LIMIT 5"
)
df_respondents = pd.DataFrame(respondents)
print("Meest recente respondenten:")
display(df_respondents)

# Use the most recent respondent by default — change index to pick another
RESPONDENT_ID = respondents[0]['respondent_id']
SET_ID        = respondents[0]['set_id']
print(f"\nGebruik respondent_id={RESPONDENT_ID}, set_id={SET_ID}")

In [ ]:
# Fetch all tasks for this respondent
# Task_set has columns act1_task_1 … act1_task_N (one per task)
# We pivot them into rows: (task_number, task_id)
task_set_row = query("SELECT * FROM Task_set WHERE set_id = ?", (SET_ID,))[0]
n_tasks = sum(1 for k in task_set_row if k.startswith('act1_task_'))

rows = []
for i in range(1, n_tasks + 1):
    task_id = task_set_row[f'act1_task_{i}']
    task_data = query("SELECT * FROM Act1_task WHERE task_id = ?", (task_id,))[0]
    rows.append({
        'taak_nr':            i,
        'task_id':            task_id,
        'kruispunt_1_id':     extract_id(task_data['alt1_att1_streetview']),
        'kruispunt_2_id':     extract_id(task_data['alt2_att1_streetview']),
        'intensiteit_1':      task_data['alt1_att2_intensiteit'],
        'intensiteit_2':      task_data['alt2_att2_intensiteit'],
        'url_1':              task_data['alt1_att1_streetview'],
        'url_2':              task_data['alt2_att1_streetview'],
    })

df_tasks = pd.DataFrame(rows)
print(f"{n_tasks} taken gevonden.")
display(df_tasks[['taak_nr','task_id','kruispunt_1_id','intensiteit_1','kruispunt_2_id','intensiteit_2']])

In [ ]:
# Show photos for all tasks — two columns side by side (kruispunt 1 | kruispunt 2)
for _, row in df_tasks.iterrows():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(
        f"Taak {row['taak_nr']}  (task_id={row['task_id']})",
        fontsize=13, fontweight='bold'
    )

    for ax, url, label, intensiteit in [
        (axes[0], row['url_1'], f"Kruispunt #1\nID: {row['kruispunt_1_id']}", row['intensiteit_1']),
        (axes[1], row['url_2'], f"Kruispunt #2\nID: {row['kruispunt_2_id']}", row['intensiteit_2']),
    ]:
        path = url_to_path(url)
        if os.path.exists(path):
            ax.imshow(mpimg.imread(path))
        else:
            ax.text(0.5, 0.5, 'Foto niet gevonden', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f"{label}\nIntensiteit: {intensiteit}", fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Zoek een specifieke taak op basis van taaknummer (zoals zichtbaar in de URL /act1/6)
TAAK_NR = 6  # <-- pas aan

row = df_tasks[df_tasks['taak_nr'] == TAAK_NR].iloc[0]
print(f"Taak {TAAK_NR}:")
print(f"  Kruispunt #1 — ID: {row['kruispunt_1_id']}, intensiteit: {row['intensiteit_1']}")
print(f"  Kruispunt #2 — ID: {row['kruispunt_2_id']}, intensiteit: {row['intensiteit_2']}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Taak {TAAK_NR}  (task_id={row['task_id']})", fontsize=13, fontweight='bold')

for ax, url, label, intensiteit in [
    (axes[0], row['url_1'], f"Kruispunt #1 — ID: {row['kruispunt_1_id']}", row['intensiteit_1']),
    (axes[1], row['url_2'], f"Kruispunt #2 — ID: {row['kruispunt_2_id']}", row['intensiteit_2']),
]:
    path = url_to_path(url)
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
    else:
        ax.text(0.5, 0.5, 'Foto niet gevonden', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(f"{label}\nIntensiteit: {intensiteit}", fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()